# Run matrix — cola de experimentos

Notebook para Colab que define listas de `ExperimentConfig` y las corre secuencialmente con `run_matrix()`: salta las que ya estan completas, reanuda (sin borrar nada) las que quedaron incompletas de una corrida anterior -- solo recalculando, region por region, lo que falte o este invalido (ver `docs/CHECKPOINT_RESUME.md`) --, continua si una falla, y al final muestra un resumen de completadas/fallidas y MAPE promedio.

## Tres fases, independientes entre si

Este notebook queda organizado en **3 fases**, cada una en su propio bloque de celdas (config + lanzamiento). **Correr una fase NO dispara las otras** -- son bloques de celdas separados, cada uno arma su propia lista de `ExperimentConfig` y llama a `run_matrix()` por su cuenta. Podes correr solo la celda de Setup + solo la fase que te interese.

- **FASE 1A -- Modelos principales**: `xgboost`, `lightgbm`, `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`. Es la matriz baseline ya confirmada como 6/6 completa en Drive -- las configs de esta fase son **exactamente** las que ya se corrieron (mismas exogenas, mismo `train_hours`), sin ningun cambio. Volver a correr esta fase debe detectar todo como completo y saltarlo entero (ver la nota en su seccion).
- **FASE 1B -- Modelos adicionales / baselines**: `naive`, `naive_trend`, `naive_trend_seasonal`, `ar`, `ar_resid_trend_seasonal`, `lstm_resid`. Los 6 modelos migrados en la tarea de checkpoint/resume, cada uno con su config vigente en `runner.MODEL_DEFAULTS` (sin inventar ningun parametro nuevo).
- **FASE 2 -- Exogenas individuales**: para cada modelo **multivariado** (el que tenga al menos una exogena en su catalogo), una corrida por cada una de `Temperatura, IGAE, Generacion, Importacion, Exportacion`, **sin acumular** (nunca dos exogenas juntas). Generada automaticamente con `build_individual_exog_matrix()` -- no hay que escribir cada `ExperimentConfig` a mano. Los modelos univariados se excluyen solos (no generan corridas redundantes).

Ninguna fase modifica ni borra los resultados de las otras: cada `ExperimentConfig` distinto (modelo + exogenas + train_hours + forecast_horizon) cae en su propio `RUN_NAME`/carpeta determinista (`build_run_name()`, ver `config.py`), asi que 1A/1B/2 nunca se pisan entre si ni pisan lo que ya hay en Drive.

## Datos de entrada

Los 34 archivos (`IGAE_2.xlsx`, `Temperaturas promedio.csv`, y por cada una de las 8 regiones -- BCA, CEN, NES, NOR, NTE, OCC, ORI, PEN -- `{REGION}_long.csv`, `{REGION}_GEN.csv`, `{REGION}_IMP.csv`, `{REGION}_EXP.csv`) viven permanentemente en Google Drive, en `MyDrive/Bases de datos Tesis` (constante `DATA_DIR` mas abajo). Detalle exacto de columnas/formato: [`docs/DATOS_REQUERIDOS.md`](../docs/DATOS_REQUERIDOS.md).

## Modelos disponibles (los 12 registrados hoy en `runner.MODEL_DEFAULTS`/`MODEL_RUNNERS`)

`xgboost`, `lightgbm` (adaptado, ver [`docs/MODELOS_MIGRADOS.md`](../docs/MODELOS_MIGRADOS.md)), `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`, `naive`, `naive_trend`, `naive_trend_seasonal`, `ar`, `ar_resid_trend_seasonal`, `lstm_resid`. Todos procesan las 8 regiones en una sola corrida.

## Setup (una sola celda): montar Drive, instalar dependencias, cargar el proyecto

Cubre las dependencias de las 3 fases (`optuna`/`lightgbm`, que no vienen preinstaladas en el runtime estandar de Colab; `tensorflow`/`statsmodels`/`xgboost`/`pandas`/`numpy`/`scikit-learn` si vienen). Corre esta celda una sola vez por sesion, sin importar que fase(s) vayas a lanzar despues.

In [ ]:
import os
import sys

from google.colab import drive
drive.mount("/content/drive")

# xgboost, tensorflow, statsmodels, pandas, numpy, scikit-learn ya vienen
# preinstalados en el runtime estandar de Colab; optuna y lightgbm no.
!pip install -q optuna lightgbm

REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("\nSetup completo.")
print("SRC_DIR en sys.path:", SRC_DIR in sys.path)

In [ ]:
# Ruta donde viven permanentemente los datos de entrada en Google Drive.
# Compartida por las 3 fases -- ninguna la sobreescribe ni la necesita
# distinta.
DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"

from tesis_forecast.config import ExperimentConfig
from tesis_forecast.matrix import build_individual_exog_matrix, run_matrix, resumen_dataframe

---
## FASE 1A -- Modelos principales

`xgboost`, `lightgbm`, `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`. **Estas son exactamente las configs que ya se corrieron** para esta matriz (mismas exogenas, mismo `train_hours` que la corrida ya confirmada como 6/6 completa en Drive) -- no se cambio nada aqui, solo se le puso nombre de fase.

Deja `exogenas`/`train_hours` explicitos (en vez de `None`) a proposito, para que quede documentado en el propio notebook cual es la config real de cada corrida de esta fase, no solo "el default de turno" del modulo.

**Reanudar/re-lanzar esta fase**: si volves a correr la celda de "Lanzar FASE 1A" sobre una matriz ya completa, `run_matrix()` llama `validar_resultado(run_dir)` por cada config *antes* de tocar nada -- si encuentra las 8 regiones con metricas validas, imprime `YA COMPLETO, se salta: <RUN_NAME>` y pasa a la siguiente config sin ejecutar ningun modelo. Si alguna quedara incompleta (por ejemplo la sesion de Colab se desconecto a mitad de una region), la reanuda en la misma carpeta -- el checkpoint por region (`checkpoint.cargar_checkpoint_regiones`, invocado dentro de cada `run()`) salta las regiones que ya tengan resultado valido y solo recalcula las que falten. Ningun resultado existente se borra ni se sobreescribe por volver a correr esta celda.

In [ ]:
configs_1a = [
    ExperimentConfig(
        modelo="xgboost",
        exogenas=["Temperatura", "Primarias", "Secundarias", "Terciarias", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=336,
        notas="Fase 1A -- config vigente.",
    ),
    ExperimentConfig(
        modelo="lightgbm",
        exogenas=["Temperatura", "Primarias", "Secundarias", "Terciarias", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=336,
        notas="Fase 1A -- adaptado desde celda 46, no extraccion exacta.",
    ),
    ExperimentConfig(
        modelo="lstm_direct",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=2160,
        notas="Fase 1A -- config vigente.",
    ),
    ExperimentConfig(
        modelo="sarimax",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=1440,
        notas="Fase 1A -- config vigente, orden SARIMAX fijo (sin tuning).",
    ),
    ExperimentConfig(
        modelo="fcnn",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=3600,
        notas="Fase 1A -- config vigente, produce 2 modelos por region (directa + STL-residuos).",
    ),
    ExperimentConfig(
        modelo="ensemble_stl",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=3600,
        notas="Fase 1A -- config vigente, es el pipeline mas pesado (2 redes + barrido AR por region).",
    ),
]

print(f"FASE 1A: {len(configs_1a)} configs")
configs_1a

### Lanzar FASE 1A

Corre secuencialmente. Puede tardar horas (Ensemble y FCNN entrenan redes por cada una de las 8 regiones). Si ya esta todo completo en Drive, esta celda debe correr rapido e imprimir `Saltadas: 6 (ya estaban completas)` en el resumen final, sin re-entrenar nada.

In [ ]:
resultados_1a = run_matrix(configs_1a, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_1a)

---
## FASE 1B -- Modelos adicionales / baselines

`naive`, `naive_trend`, `naive_trend_seasonal`, `ar`, `ar_resid_trend_seasonal`, `lstm_resid`. Los 6 modelos migrados en la tarea de checkpoint/resume (ver `docs/CHECKPOINT_RESUME.md` y `docs/MODELOS_MIGRADOS.md`).

`exogenas`/`train_hours`/`forecast_horizon`/`optuna_n_trials` se dejan en `None` a proposito: esa es la señal para que `resolve_run()`/`run_experiment()` usen exactamente lo que ya esta registrado en `runner.MODEL_DEFAULTS` para cada modelo -- nada se inventa ni se fuerza aqui. Los primeros 5 son univariados (`catalogo == []` en su modulo, `resolve_run()` rechaza cualquier exogena que se les pase); `lstm_resid` es multivariado y usa su catalogo de 5 exogenas por defecto (`Temperatura, IGAE, Generacion, Importacion, Exportacion`).

Misma logica de reanudacion/checkpoint que la Fase 1A: una config ya completa se salta, una incompleta se reanuda solo desde las regiones que falten.

In [ ]:
configs_1b = [
    ExperimentConfig(modelo="naive",
                      notas="Fase 1B -- Naive."),
    ExperimentConfig(modelo="naive_trend",
                      notas="Fase 1B -- Naive + Tendencia."),
    ExperimentConfig(modelo="naive_trend_seasonal",
                      notas="Fase 1B -- Naive + Tendencia + Estacionalidad."),
    ExperimentConfig(modelo="ar",
                      notas="Fase 1B -- AR standalone."),
    ExperimentConfig(modelo="ar_resid_trend_seasonal",
                      notas="Fase 1B -- AR sobre residuos + Tendencia + Estacionalidad."),
    ExperimentConfig(modelo="lstm_resid",
                      notas="Fase 1B -- LSTM multivariada sobre residuos + Tendencia + Estacionalidad."),
]

print(f"FASE 1B: {len(configs_1b)} configs")
configs_1b

### Lanzar FASE 1B

In [ ]:
resultados_1b = run_matrix(configs_1b, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_1b)

---
## FASE 2 -- Exogenas individuales

Para cada modelo **multivariado** (el que tenga al menos una exogena en su catalogo -- ver `runner.MODEL_DEFAULTS`), una corrida por cada exogena de `EXOGENAS_INDIVIDUALES`, **sin acumular** (nunca dos exogenas juntas en la misma corrida). Los modelos univariados (hoy: `naive`, `naive_trend`, `ar`, `naive_trend_seasonal`, `ar_resid_trend_seasonal`) se excluyen automaticamente -- no generan corridas redundantes con una sola exogena c/u.

Cada corrida usa exactamente los defaults cientificos vigentes del modelo (`train_hours`/`forecast_horizon`/`optuna_n_trials`, ver `runner.MODEL_DEFAULTS`); lo UNICO que cambia respecto a las Fases 1A/1B es `exogenas`. El `RUN_NAME` de cada corrida queda en una carpeta propia (ej. `XGBoost_train336h_fh168h_Temp` vs. `XGBoost_train336h_fh168h_IGAE` vs. el baseline de la Fase 1A `XGBoost_train336h_fh168h_Temp-Prim-Sec-Terc-IGAE-Gen-Imp-Exp`), asi que esta fase **no toca ni sobreescribe** los resultados de 1A ni 1B. El checkpoint por region aplica exactamente igual: una corrida individual completa se salta, una incompleta se reanuda solo desde las regiones faltantes.

Ver `tests/test_individual_exog_matrix.py` para la verificacion (numero de configs, sin colisiones de RUN_NAME, defaults preservados, exclusiones documentadas, etc.).

In [ ]:
EXOGENAS_INDIVIDUALES = [
    "Temperatura",
    "IGAE",
    "Generacion",
    "Importacion",
    "Exportacion",
]

configs_individuales = build_individual_exog_matrix(exogenas=EXOGENAS_INDIVIDUALES)

print(f"FASE 2: {len(configs_individuales)} configs")
print("\nPor modelo:")
for modelo in sorted({c.modelo for c in configs_individuales}):
    exogenas_de_modelo = sorted(c.exogenas[0] for c in configs_individuales if c.modelo == modelo)
    print(f"  {modelo}: {exogenas_de_modelo}")

configs_individuales

### Lanzar FASE 2

In [ ]:
resultados_individuales = run_matrix(configs_individuales, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_individuales)

---
## Auditoria (solo lectura) -- que de la Fase 2 ya existe en `Pipeline_Resultados`

Antes de lanzar la Fase 2 (o de duplicar corridas), esta seccion compara las 35 configs que produce HOY `build_individual_exog_matrix()` contra TODO lo que ya hay en `Pipeline_Resultados/` -- incluidas carpetas de corridas anteriores a `runner.py`/al esquema `RUN_NAME` actual, que no tienen `config.json`. **Es 100% de solo lectura: no ejecuta ningun modelo, no mueve/renombra/borra/sobreescribe absolutamente nada.**

La identidad de cada carpeta (modelo, exogenas, `train_hours`, `forecast_horizon`) se determina por CONTENIDO (leyendo `config.json` si existe, o infiriendo de `metricas.csv`/`config_usada.csv` si no existe), nunca solo por el nombre de la carpeta. Ver el docstring de `src/tesis_forecast/audit.py` para el detalle exacto de como se infiere una carpeta legacy y que se hace cuando algo no se puede determinar con confianza (nunca se adivina: queda marcado en `razon` para revision manual).

Cada una de las 35 configs queda clasificada como:
- `exact_match_complete`: ya hay un resultado cientificamente equivalente y completo -- reutilizable tal cual.
- `exact_match_partial`: existe pero le faltan regiones/archivos.
- `legacy_not_equivalent`: existe algo con el mismo modelo+exogena pero con `train_hours`/`forecast_horizon` distinto -- no es intercambiable sin alterar la comparabilidad.
- `missing`: no se encontro nada.

Esta celda **no migra ni convierte nada** -- solo genera el inventario para decidir que vale la pena reutilizar antes de correr la Fase 2.

In [ ]:
from tesis_forecast.audit import auditar_matriz_individual_actual

# Misma lista que usa la Fase 2 -- se repite aca (en vez de depender de
# que esa celda ya se haya corrido) para que esta seccion siga siendo
# independiente de las demas, igual que las Fases 1A/1B/2.
EXOGENAS_INDIVIDUALES_AUDITORIA = ["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"]

inventario_df = auditar_matriz_individual_actual(
    pipeline_resultados_dir="/content/drive/MyDrive/Pipeline_Resultados",
    exogenas=EXOGENAS_INDIVIDUALES_AUDITORIA,
    regiones_esperadas=None,  # None = las 8 regiones de siempre (REGIONS_ALL)
)

inventario_df

### (Opcional) Guardar el inventario para revisarlo aparte

In [ ]:
inventario_path = "/content/drive/MyDrive/Pipeline_Resultados/auditoria_exogenas_individuales.csv"
inventario_df.to_csv(inventario_path, index=False, encoding="utf-8-sig")
print(f"Inventario guardado en: {inventario_path}")

---
## FASE 3 -- Temperatura + IGAE (las 2 juntas, sin acumular con nada mas)

Los 7 modelos multivariados, cada uno con **exactamente** `["Temperatura", "IGAE"]` como exogenas -- no es una matriz generada, son 7 `ExperimentConfig` explicitos (a proposito: esto no necesita `build_individual_exog_matrix()`, que genera 1 exogena a la vez, ni ninguna maquinaria nueva). El resto de los parametros (`train_hours`, `forecast_horizon`, `optuna_n_trials`) quedan en `None` para que `resolve_run()` use exactamente los defaults vigentes de cada modelo en `runner.MODEL_DEFAULTS` -- los mismos que usa la Fase 1A.

**Pensada para correr en un Colab DISTINTO** del que este corriendo la Fase 2 (35 corridas) -- ver la seccion "Colab aparte -- FASE 3" mas abajo para las celdas exactas. Ambas fases escriben en el mismo Drive/`Pipeline_Resultados`, pero en `RUN_NAME` distintos (`..._Temp-IGAE` vs. `..._Temp`/`..._IGAE`/etc. de la Fase 2, o el catalogo completo de la Fase 1A) -- nunca se pisan entre si.

### Resume/checkpoint -- sin cambios en la logica existente

`resolve_run()` calcula el `RUN_NAME` de cada config de forma **determinista** (modelo + train_hours + forecast_horizon + exogenas, sin timestamp ni azar) -- por eso, sin importar en que Colab o en que momento se corra esta celda, FCNN con `exogenas=["Temperatura","IGAE"]` siempre resuelve al mismo `RUN_NAME`: `FCNN_train3600h_fh168h_Temp-IGAE`. Si esa carpeta ya existe en Drive (completa o incompleta, generada por OTRO Colab corriendo esta misma Fase 3), `run_matrix()`/`run_experiment()` usan exactamente el mismo mecanismo de siempre (`validar_resultado()` + `checkpoint.cargar_checkpoint_regiones()`, sin ningun cambio ni caso especial):

- si esa carpeta ya esta completa (8/8 regiones) cuando corras esta celda, se salta entera;
- si esta parcial, se reanuda SOLO desde las regiones que falten -- las que ya esten guardadas no se recalculan;
- ninguna corrida de Fase 3 crea un `RUN_NAME` distinto para FCNN: siempre es el mismo, por diseno (no hay forma de que `resolve_run()` genere otro nombre para la misma config).

**Precaucion (no es un problema de checkpoint, es de correr dos procesos a la vez):** si otro Colab esta **en este momento** escribiendo activamente en `FCNN_train3600h_fh168h_Temp-IGAE` (a mitad de una region), no corras la celda "Lanzar FASE 3" hasta que termine o se detenga -- dos procesos escribiendo la misma carpeta al mismo tiempo pueden pisarse entre si. Si ya termino (completa o parcial-y-abandonada), correr esta celda es seguro: la salta o la reanuda segun corresponda.

Esta celda **no dispara la Fase 1A, 1B ni 2** -- son bloques de celdas totalmente separados.

In [ ]:
# 1. CONSTRUIR FASE 3
configs_temp_igae = [
    ExperimentConfig(modelo="xgboost", exogenas=["Temperatura", "IGAE"]),
    ExperimentConfig(modelo="lightgbm", exogenas=["Temperatura", "IGAE"]),
    ExperimentConfig(modelo="lstm_direct", exogenas=["Temperatura", "IGAE"]),
    ExperimentConfig(modelo="sarimax", exogenas=["Temperatura", "IGAE"]),
    ExperimentConfig(modelo="fcnn", exogenas=["Temperatura", "IGAE"]),
    ExperimentConfig(modelo="ensemble_stl", exogenas=["Temperatura", "IGAE"]),
    ExperimentConfig(modelo="lstm_resid", exogenas=["Temperatura", "IGAE"]),
]

print(f"FASE 3: {len(configs_temp_igae)} configs\n")
for c in configs_temp_igae:
    print(f"  {c.modelo:15s} exogenas={c.exogenas}")

### 2. PREVIEW FASE 3 (antes de ejecutar)

Usa `resolve_run()`/`build_run_name()` reales (no numeros copiados a mano) para mostrar, de cada una de las 7 configs, exactamente que va a usar `run_experiment()` si se lanza: `train_hours`/`forecast_horizon`/`optuna_n_trials` resueltos y el `RUN_NAME` determinista. **No ejecuta ningun modelo ni toca Drive** -- `resolve_run()` es puro calculo.

In [ ]:
# 2. PREVIEW FASE 3 -- resolve_run() real, sin ejecutar nada
from tesis_forecast.runner import resolve_run
import pandas as pd

filas_preview = []
for c in configs_temp_igae:
    r = resolve_run(c)
    filas_preview.append({
        "modelo": c.modelo,
        "exogenas": c.exogenas,
        "train_hours_resuelto": r.train_hours,
        "forecast_horizon_resuelto": r.forecast_horizon,
        "optuna_n_trials_resuelto": r.optuna_n_trials,
        "run_name": r.run_name,
    })

preview_df = pd.DataFrame(filas_preview)
preview_df

### 3. EJECUTAR FASE 3

In [ ]:
# 3. EJECUTAR FASE 3 -- exclusivamente run_matrix() sobre configs_temp_igae + resumen_dataframe()
resultados_temp_igae = run_matrix(configs_temp_igae, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_temp_igae)

---
## CONSOLIDAR RESULTADOS PARA LOOKER STUDIO

Seccion independiente de las 3 fases + Fase 3 de arriba -- **no entrena ni corre ningun experimento**, solo lee lo que ya esta escrito en `Pipeline_Resultados/` y arma tres CSV planos (`metricas_master.csv`, `series_master.csv`, `reporte_consolidacion.csv`) pensados para conectar Looker Studio directamente sobre Drive. Se ubica al final a proposito: pensada para correrse despues de que 1A/1B/2/3 (o cualquier subconjunto de ellas) ya tengan resultados en Drive, pero podes correrla en cualquier momento, incluso con corridas todavia en progreso en otro Colab -- el agregador solo incluye corridas COMPLETAS y reporta cuantas ignoro por estar incompletas (ver `src/tesis_forecast/aggregator.py`).

Es de solo lectura sobre las carpetas de corridas: nunca modifica ni borra ningun `series.csv`/`metricas.csv`/`config.json` existente. Reejecutarla es seguro e idempotente -- cada corrida reconstruye los 3 CSV desde cero a partir de `Pipeline_Resultados/` (nunca acumula sobre su propia salida anterior ni depende de que los CSV ya existan), asi que no hay riesgo de filas duplicadas por correrla varias veces.

- **`metricas_master.csv`**: una fila por cada fila de `metricas.csv` de cada corrida completa (misma granularidad region/modelo de siempre -- FCNN sigue aportando 2 filas por region), con metadata agregada (`run_name`, `modelo` canonico, `familia_experimento`, `exogenas`, `exogena_individual`, `train_hours`, `forecast_horizon`, etc.). MAE/RMSE/MAPE/sMAPE nunca se recalculan.
- **`series_master.csv`**: predicciones de cada corrida como filas independientes, mas los valores reales del horizonte comparable **una sola vez por region+timestamp** (no se concatena `series.csv` a lo bruto), excluyendo las filas `componente_pred` internas de Ensemble STL.
- **`reporte_consolidacion.csv`**: inventario de TODAS las carpetas consideradas (incluidas y excluidas), con `run_name`/`familia`/`estado`/`incluido`/`razon` -- la trazabilidad de que entro a los masters y que no, y por que.

`familia_experimento` (`1A`/`1B`/`individual`/`temp_igae`) se deriva de `modelo`+`exogenas` de `config.json` de cada corrida, nunca del nombre de la carpeta -- Fase 3 (Temperatura+IGAE) clasifica como `temp_igae`, no se confunde con `1A` aunque el modelo (ej. xgboost) tambien exista en esa familia.

Los 3 CSV se escriben en `Pipeline_Resultados/Consolidado/` (subcarpeta dedicada, nunca mezclada con las carpetas `<RUN_NAME>/` de las corridas -- el propio descubrimiento la excluye explicitamente para no intentar leerla como si fuera una corrida).

In [ ]:
# A. IMPORTAR
from tesis_forecast.aggregator import consolidar_resultados

In [ ]:
# B. EJECUTAR CONSOLIDACION
# Misma carpeta donde run_experiment()/run_matrix() ya escriben cada
# corrida (io_drive.resolve_base_dir()). La salida (metricas_master.csv,
# series_master.csv, reporte_consolidacion.csv) se escribe en la
# subcarpeta Consolidado/ -- no hace falta crearla, consolidar_resultados()
# la crea si no existe.
PIPELINE_RESULTADOS_DIR = "/content/drive/MyDrive/Pipeline_Resultados"

metricas_master_df, series_master_df, descubrimiento = consolidar_resultados(
    pipeline_resultados_dir=PIPELINE_RESULTADOS_DIR,
    regiones_esperadas=None,  # None = las 8 regiones de siempre (REGIONS_ALL)
)

In [ ]:
# C. RESUMEN
n_excluidos = descubrimiento.n_incompletos + descubrimiento.n_sin_config

print(f"Runs incluidos:  {len(descubrimiento.runs)}")
print(f"Runs excluidos:  {n_excluidos} ({descubrimiento.n_incompletos} incompletos, {descubrimiento.n_sin_config} sin config.json)")
print(f"Filas metricas_master: {len(metricas_master_df):,}")
print(f"Filas series_master:   {len(series_master_df):,}")

print("\nConteo por familia:")
print(metricas_master_df.drop_duplicates("run_name")["familia_experimento"].value_counts())

print("\nConteo por modelo:")
print(metricas_master_df.drop_duplicates("run_name")["modelo"].value_counts())

In [ ]:
# D. RUTAS FINALES
import os

CONSOLIDADO_DIR = os.path.join(PIPELINE_RESULTADOS_DIR, "Consolidado")

print("metricas_master.csv:       ", os.path.join(CONSOLIDADO_DIR, "metricas_master.csv"))
print("series_master.csv:         ", os.path.join(CONSOLIDADO_DIR, "series_master.csv"))
print("reporte_consolidacion.csv: ", os.path.join(CONSOLIDADO_DIR, "reporte_consolidacion.csv"))